<a href="https://colab.research.google.com/github/areejtechcampus/0x01.c/blob/master/05_semantic_searchAssessment5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/agentic-ai-systems/blob/main/Lessons/L01/05_semantic_search.ipynb)

# Semantic Search: Indexing and Retrieval

[Source: Build a semantic search engine with LangChain | LangChain](https://docs.langchain.com/oss/python/langchain/knowledge-base).

Here we will build a search engine over a PDF document. This will allow us to retrieve passages in the PDF that are similar to an input query.

This guide focuses on LangChain's abstractions related to indexing and retrieval of text data:

* [Documents and document loaders](https://docs.langchain.com/oss/python/integrations/document_loaders);
* [Text splitters](https://docs.langchain.com/oss/python/integrations/splitters);
* [Embeddings](https://docs.langchain.com/oss/python/integrations/text_embedding);
* [Vector stores](https://docs.langchain.com/oss/python/integrations/vectorstores) and [retrievers](https://docs.langchain.com/oss/python/integrations/retrievers).

1. **Indexing**
   1. **Input processing** – Transform raw data into structured documents
   2. **Embedding & storage** – Convert text into searchable vector representations
2. **Retrieval** – Find relevant information based on user queries

This *Relevant context* would later be augmented (added) into an LLM's context window, allowing the LLM to answer questions based on data. The term for this kind of interaction is called: **Retrieval Augmented Generation (RAG)**.


## Setup


### Installation

This tutorial requires the `langchain-community` and `pypdf` packages. Using [uv](https://docs.astral.sh/uv/):

```bash
%pip install langchain-community pypdf
```


For more details, see our [Installation guide](https://docs.langchain.com/oss/python/langchain/install).


## 1. Documents and document loaders

LangChain implements a [Document](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) abstraction, which is intended to represent a unit of text and associated metadata. It has three attributes:

* `page_content`: a string representing the content;
* `metadata`: a dict containing arbitrary metadata;
* `id`: (optional) a string identifier for the document.

The `metadata` attribute can capture information about the source of the document, its relationship to other documents, and other information. Note that an individual [`Document`](https://reference.langchain.com/python/langchain_core/documents/#langchain_core.documents.base.Document) object often represents a chunk of a larger document.

We can generate sample documents when desired:

In [ ]:
!pip install faiss-cpu langchain-community langchain-huggingface pypdf sentence-transformers

In [ ]:
import os
import asyncio


import nest_asyncio
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


nest_asyncio.apply()

def load_documents(file_path):
    print(f"--- 1. Loading: {file_path} ---")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} was not found.")

    if file_path.endswith('.pdf'):
        loader = PyPDFLoader(file_path)
    else:
        loader = TextLoader(file_path)

    docs = loader.load()

    #Chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    splits = text_splitter.split_documents(docs)
    print(f"Done loading. Created {len(splits)} chunks.")
    return splits

def create_vector_store(splits):
    print("--- 2 & 3. Embedding and Storing ---")
    # Embeddings
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # Vesctorstore
    vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
    return vectorstore

async def perform_search(vectorstore, query):
    print(f"\n--- 4. Searching for: '{query}' ---")

    # Synchronous Search
    print("\n[Sync Search Result]:")
    results = vectorstore.similarity_search(query, k=1)
    if results:
        print(f"Content: {results[0].page_content[:200]}...")
        print(f"Source: Page {results[0].metadata.get('page', 'N/A')}")

    # Asynchronous Search
    print("\n[Async Search Result]:")
    async_results = await vectorstore.asimilarity_search(query, k=1)
    if async_results:
        print(f"Content: {async_results[0].page_content[:200]}...")

    #
    for i, res in enumerate(results):
        print(f"\nDetailed Result {i+1}:")
        print("-" * 20)
        print(res.page_content)
        print(f"Source: Page {res.metadata.get('page', 'N/A')}")


file_name = "/content/aws-overview.pdf"
user_query = "What is the main topic of this document?"

async def run_pipeline():
    try:

        chunks = load_documents(file_name)


        db = create_vector_store(chunks)


        await perform_search(db, user_query)

    except Exception as e:
        print(f"Error: {e}. Make sure the file exists in your directory.")


if __name__ == "__main__":
    asyncio.run(run_pipeline())

--- 1. Loading: /content/aws-overview.pdf ---
Done loading. Created 689 chunks.
--- 2 & 3. Embedding and Storing ---


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- 4. Searching for: 'What is the main topic of this document?' ---

[Sync Search Result]:
Content: Application Integration
Topics
14...
Source: Page 20

[Async Search Result]:
Content: Application Integration
Topics
14...

Detailed Result 1:
--------------------
Application Integration
Topics
14
Source: Page 20



However, the LangChain ecosystem implements [document loaders](https://docs.langchain.com/oss/python/langchain/retrieval#document-loaders) that [integrate with hundreds of common sources](https://docs.langchain.com/oss/python/integrations/document_loaders/). This makes it easy to incorporate data from these sources into your AI application.
